<center><h1>Building the Antimony Model</h1></center>
<hr/>

Assembles a complete [Antimony](https://antimony.sourceforge.net/) model string from the PSS
network and the pre-computed mapper `.pkl` files.

| Section | Content |
|---|---|
| `global` | Shared constants (`n`, `k_deg`) |
| `compartments` | One entry per unique compartment |
| `species` | Species declarations with compartment assignment |
| `reactions` | Stoichiometry lines with rate reference (`v_<id>`) |
| `rate_laws` | Kinetic rate-law expressions (assignment rules) |
| `decay_rates` | First-order degradation reactions (one per species) |
| `concentrations` | Initial concentration assignments (one per species) |
| `kinetics` | `kcat_*` / `km_*` / `Kd_*` parameter values |
| `events` | Stimulus pulses |

In [1]:
import json
import re
from pathlib import Path
from typing import Optional

import dill
import numpy as np
import pandas as pd

from scripts.utils import (
    PSS_REACTIONS_PATH,
    PSS_SPECIES_PATH,
    PSS_DECAY_RATE_MAPPER,
    PSS_INITIAL_CONCENTRATION_MAPPER,
    KINETIC_RATE_LAWS_PATH,
    KINETIC_RATE_LAWS_PATH_MAS,
    PSS_KINETIC_PARAMETER_MAPPER,
    REACTIONS_BY_COMBINATION_PATH,
    REACTIONS_BY_COMBINATION_SHEET_MAS,
    ANTIMONY_PATH,
    RED, RESET,
)

## Load

In [2]:
reactions = json.loads(Path(PSS_REACTIONS_PATH).read_text())
species   = json.loads(Path(PSS_SPECIES_PATH).read_text())

print(f"Loaded {RED}{len(species)}{RESET} species and {RED}{len(reactions)}{RESET} reactions.")

Loaded 1139 species and 864 reactions.


In [4]:
decay_rate_mapper    = dill.load(open(PSS_DECAY_RATE_MAPPER,            'rb'))
concentration_mapper = dill.load(open(PSS_INITIAL_CONCENTRATION_MAPPER, 'rb'))
rate_law_mapper      = dill.load(open(KINETIC_RATE_LAWS_PATH,           'rb'))
kinetics_mapper      = dill.load(open(PSS_KINETIC_PARAMETER_MAPPER,     'rb'))

### Mapper examples

In [5]:
specie_ex = 's_VPg_cyt_p'
rxn_ex    = 'rx00669'

print(reactions['transport_MYB334465_pa_cyt_to_nuc'])
print(reactions[rxn_ex])
print(species[specie_ex])
print("decay      :", decay_rate_mapper(specie_ex))
print("conc       :", concentration_mapper(specie_ex))
print("rate law   :", rate_law_mapper(rxn_ex))
print("kinetics   :", kinetics_mapper(rxn_ex, reactions))

{'reactants': ['s_MYB334465_cyt_pa'], 'products': ['s_MYB334465_nuc_pa'], 'modifiers': []}
{'reactants': ['s_VPg_cyt_p', 's_LHB1B1_cyt_p'], 'products': ['s_LHB1B1VPg_cyt_c'], 'modifiers': []}
{'name': 'VPg', 'compartment': 'c_cyt', 'type': 'protein'}
decay      : None
conc       : 1.06674414743483e-06
rate law   : (((kcat*s_VPg_cyt_p*s_LHB1B1_cyt_p)/(Kd_s_VPg_cyt_p*Kd_s_LHB1B1_cyt_p))-(s_LHB1B1VPg_cyt_c/(Kd_s_LHB1B1VPg_cyt_c*kcat)))/((1+s_VPg_cyt_p/Kd_s_VPg_cyt_p)*(1+s_LHB1B1_cyt_p/Kd_s_LHB1B1_cyt_p)+(1+s_LHB1B1VPg_cyt_c/Kd_s_LHB1B1VPg_cyt_c) - 1)
kinetics   : {'s_VPg_cyt_p': (0.00036, 1.02885), 's_LHB1B1_cyt_p': (0.00036, 1.02885)}


## Reaction Type Map

In [6]:
excel = pd.read_excel(
    REACTIONS_BY_COMBINATION_PATH,
    sheet_name=REACTIONS_BY_COMBINATION_SHEET_MAS,
    skiprows=0,
)

reaction_type_map: dict[str, str] = {
    rx.strip(): row['reaction_type']
    for _, row in excel.iterrows()
    for rx in row['reaction_ids'].split(',')
}

print(f"Reaction types: {RED}{set(reaction_type_map.values())}{RESET}")

Reaction types: {'catalysis', 'binding/oligomerisation', 'dissociation', 'unknown', 'protein deactivation', 'transcriptional/translational activation', 'degradation/secretion', 'translocation', 'transcriptional/translational repression', 'protein activation', 'transport'}


## Helpers

In [7]:
def extract_rate_components(expr: str) -> dict[str, list[str]]:
    """Extract unique parameter tokens from a kinetic rate expression.

    Args:
        expr: RHS kinetic rate expression string.

    Returns:
        Dict mapping token type → list of unique matched tokens (order preserved).
        Keys: 'kcat', 'km', 'Kd', 'vmax', 'kmc', 'kic', 'vmag', 'kG',
              'substrates', 'compartments'.
    """
    def unique(seq: list[str]) -> list[str]:
        seen: set[str] = set()
        result = []
        for x in seq:
            if x not in seen:
                seen.add(x)
                result.append(x)
        return result

    patterns = {
        'kcat':  r'kcat_[A-Za-z0-9_]+|\bkcat\b',
        'km':    r'km_[A-Za-z0-9_]+|\bkm\b',
        'Kd':    r'Kd_[A-Za-z0-9_]+',
        'vmax':  r'vmax_[A-Za-z0-9_]+',
        'kmc':   r'kmc_[A-Za-z0-9_]+',
        'kic':   r'kic_[A-Za-z0-9_]+',
        'vmag':  r'vmag_[A-Za-z0-9_]+',
        'kG':    r'kG_[A-Za-z0-9_]+',
    }

    components = {key: unique(re.findall(pat, expr)) for key, pat in patterns.items()}
    components['substrates']   = unique(re.findall(r'\b(s\d+)\b', expr))
    components['compartments'] = unique(re.findall(r'\b(c\d+)\b', expr))
    return components

## Build Antimony

In [8]:
def build_antimony(
    reactions_2use: list[str],
    reactions:      dict,
    species:        dict,
) -> str:
    """Assemble an Antimony model string for the given reaction subset.

    Args:
        reactions_2use: Ordered list of reaction IDs to include.
        reactions:      Full PSS reactions dict (used by kinetics_mapper).
        species:        Full PSS species dict.

    Returns:
        Complete Antimony model string ready to write to file.
    """
    reactions_all = reactions.copy()
    reactions     = {k: v for k, v in reactions.items() if k in reactions_2use}
    species_ids   = {
        s for rxn in reactions_2use if rxn in reactions
        for vals in reactions[rxn].values() for s in vals
    }
    species = {k: v for k, v in species.items() if k in species_ids}

    # ── Static sections ────────────────────────────────────────────────────────
    compartments = sorted({species[s]['compartment'] for s in species})

    sections: dict[str, str] = {
        'global': (
            "\t# Global:\n"
            "\tn = 1;\n"
            "\tk_deg = 1e-6;"
        ),
        'compartments': (
            "\t# Compartments:\n" +
            "\n".join(f"\t{c:<10} = 1;" for c in compartments)
        ),
        'species': (
            "\t# Species Definitions:\n" +
            "\n".join(
                f"\tspecies {s:<35} in {species[s]['compartment']};"
                for s in species
            )
        ),
        'reactions':      "\t# Reactions:\n",
        'rate_laws':      "\t# Rate Laws:\n",
        'decay_rates':    "\t# Decay Rates:\n",
        'concentrations': "\t# Concentrations:\n",
        'kinetics':       "\t# Kinetics:\n",
        'events': (
            "\t# Events:\n"
            "\tE1: at time >= 1000: s_JA_cyt_m = s_JA_cyt_m + 100;\n"
        ),
    }

    # ── Pre-populate concentrations & decay rates (one pass over all species) ──
    # These are species-level properties — completely independent of reactions.
    for s in species:
        conc = concentration_mapper(s)
        if conc is not None:
            sections['concentrations'] += f"\t{s:<35} = {conc};\n"

        decay = decay_rate_mapper(s)
        if decay is not None:
            sections['decay_rates'] += (
                f"\td_{s:<35}: {s:<35} -> ; "
                f"({s} * {decay}) / (1e-4 + {s});\n"
            )

    # ── Reaction-level sections ────────────────────────────────────────────────
    for rxn in reactions:
        rxn_data = reactions[rxn]

        # Gene reactants are boundary species (prefixed with $, held constant)
        reactants = [
            f"${r}" if species.get(r, {}).get('type') == 'gene' else r
            for r in rxn_data['reactants']
        ]
        products = rxn_data['products']

        sections['reactions'] += (
            f"\t{rxn}: {' + '.join(reactants):<50} -> "
            f"{' + '.join(products):<30}; v_{rxn};\n"
        )

        # Rate law — skip reaction entirely if unavailable
        try:
            rate_law = rate_law_mapper(rxn)
        except TypeError as e:
            print(f"  [WARN] {rxn}: rate_law_mapper raised {e}")
            continue
        if not rate_law:
            print(f"  [WARN] {rxn}: no rate law — skipping.")
            continue

        # Substitute generic kcat/km tokens with reaction-scoped names
        rate_law = re.sub(r'\bkcat\b', f'kcat_{rxn}', rate_law)
        rate_law = re.sub(r'\bkm\b',   f'km_{rxn}',   rate_law)
        components = extract_rate_components(rate_law)

        sections['rate_laws'] += f"\tv_{rxn} := {rate_law} + 1e-16;\n"

        # ── Kinetic parameters ─────────────────────────────────────────────────
        # kinetics_mapper returns {reactant_id: (km_mM, kcat_1s), ...}
        kinetics  = kinetics_mapper(rxn, reactions_all)
        kms       = {r: km   for r, (km,   _)    in kinetics.items() if km   is not None}
        kcats     = {r: kcat for r, (_,    kcat) in kinetics.items() if kcat is not None}
        km_min    = float(np.min(list(kms.values())))   if kms   else None
        kcat_min  = float(np.min(list(kcats.values()))) if kcats else None

        if components['kcat'] and kcat_min is not None:
            sections['kinetics'] += f"\tkcat_{rxn:<30} = {kcat_min};\n"

        for kd_token in components['Kd']:
            substrate = kd_token.split('Kd_', 1)[1]
            km_val    = kms.get(substrate, km_min)
            if km_val is not None:
                sections['kinetics'] += f"\t{kd_token:<35} = {km_val};\n"

        if components['km'] and km_min is not None:
            sections['kinetics'] += f"\tkm_{rxn:<30} = {km_min};\n"

    return "model PSS\n\n" + "\n\n".join(sections.values()) + "\nend"

## Reaction Subsets

In [9]:
reactions_ja = [
    'rx00693', 'rx00026', 'rx00029', 'rx00030', 'rx00031', 'rx00694',
    'rx00033', 'rx00034', 'rx00035', 'rx00036', 'rx00037', 'rx00038',
    'rx00039', 'rx00040', 'rx00696', 'rx00041', 'rx00042', 'rx00043',
    'rx00044', 'rx00018', 'rx00047', 'rx00028',
]
reactions_aba = [
    'rx00568', 'rx00710', 'rx00709', 'rx00748', 'rx00750', 'rx00749',
    'rx00474', 'rx00751', 'rx00752', 'rx00753', 'rx00552', 'rx00553',
    'rx00554', 'rx00549', 'rx00555', 'rx00556', 'rx00557', 'rx00558',
    'rx00559', 'rx00560',
]

## Build & Inspect

In [10]:
# Uncomment the subset to build:
# antimony = build_antimony(list(reactions.keys()), reactions, species)  # Whole PSS
antimony = build_antimony(reactions_ja,  reactions, species)             # JA pathway
# antimony = build_antimony(reactions_aba, reactions, species)           # ABA pathway

print(antimony)

model PSS

	# Global:
	n = 1;
	k_deg = 1e-6;

	# Compartments:
	c_chl      = 1;
	c_cyt      = 1;
	c_nuc      = 1;
	c_per      = 1;

	# Species Definitions:
	species s_COI1_nuc_pa                       in c_nuc;
	species s_JAZ_nuc_pa                        in c_nuc;
	species s_MYC2_nuc_pa                       in c_nuc;
	species s_SCF_cyt_ca                        in c_cyt;
	species s_ACX_per_pa                        in c_per;
	species s_ALA_chl_m                         in c_chl;
	species s_Ile_cyt_m                         in c_cyt;
	species s_13HPOT_chl_m                      in c_chl;
	species s_LOX_chl_pa                        in c_chl;
	species s_1213EOT_chl_m                     in c_chl;
	species s_AOS_chl_pa                        in c_chl;
	species s_OPDA_chl_m                        in c_chl;
	species s_AOC_chl_pa                        in c_chl;
	species s_OPDA_per_m                        in c_per;
	species s_OPC8_per_m                        in c_per;
	species s_OPR_per_

## Save

In [11]:
with open(ANTIMONY_PATH, 'w') as f:
    f.write(antimony)

print(f"Saved Antimony model → {ANTIMONY_PATH}")

Saved Antimony model → /home/ratanasoski/Desktop/Master_Thesis/ThesisRepository/results/Antimony.txt
